# 04 — Three-Class Shop Classification

Labels are 0 = Kém, 1 = Bình thường, 2 = Uy tín. Ground truth must be assigned and reviewed by people.

In [ ]:
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore")
current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "config" / "config.yaml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate project root containing config/config.yaml")
with (PROJECT_ROOT / "config" / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", OUTPUT / "powerbi", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
fingerprint_paths = [PROJECT_ROOT / "config" / "config.yaml", project_path("shops")]
for key in ("products", "reviews"):
    fingerprint_paths.extend(sorted(project_path(key).glob("shop_*.json")))
seller_candidate = project_path("seller_metrics")
for candidate in (seller_candidate, seller_candidate.with_suffix(".xlsx")):
    if candidate.exists(): fingerprint_paths.append(candidate)
digest = hashlib.sha256()
for path in fingerprint_paths:
    digest.update(str(path.relative_to(PROJECT_ROOT)).encode("utf-8")); digest.update(path.read_bytes())
INPUT_FINGERPRINT = digest.hexdigest()
print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_FINGERPRINT:", INPUT_FINGERPRINT)

In [ ]:
shops = pd.read_csv(PROCESSED / "shops_clean.csv")
auth_path = OUTPUT / "reports" / "Shop_Authenticity_Report.csv"
if not auth_path.exists(): raise FileNotFoundError("Run clustering and confirm authenticity mapping first")
auth = pd.read_csv(auth_path)
label_frame = shops.drop(columns=["Target_Label"], errors="ignore").merge(auth,on="Shop_ID",how="left",validate="one_to_one")
for column in ["Target_Label","Labeler","Label_Reason","Review_Status"]: label_frame[column] = ""
label_path = OUTPUT / "labeling" / "Data_To_Label.csv"
label_frame.to_csv(label_path,index=False,encoding="utf-8-sig")
print("Human-labeling template:",label_path)
display(label_frame.head())

In [ ]:
import joblib, matplotlib.pyplot as plt, seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

source = OUTPUT / "labeling" / "Data_Labeled.csv"
if not source.exists(): raise FileNotFoundError("Review Data_To_Label.csv and save it as output/labeling/Data_Labeled.csv")
data = pd.read_csv(source)
features = ["Years_Active","Total_Products","Follower_Count_k","Rating_Average","Chat_Response_Rate_pct","Authentic_Review_Rate","Conversion_Rate_pct"]
required_review = ["Labeler","Label_Reason","Review_Status"]
if not set(required_review) <= set(data): raise ValueError("Labeled data must preserve Labeler, Label_Reason and Review_Status")
approved = data["Review_Status"].fillna("").astype(str).str.lower().eq("approved")
documented = data[["Labeler","Label_Reason"]].fillna("").astype(str).apply(lambda s:s.str.strip().ne("")).all(axis=1)
if not (approved & documented).all(): raise ValueError("Every label must be approved and document its labeler and reason")
data = data.dropna(subset=["Target_Label",*features]).copy(); data["Target_Label"] = data["Target_Label"].astype(int)
if not set(data["Target_Label"]) <= {0,1,2}: raise ValueError("Target_Label must be 0, 1 or 2")
counts = data["Target_Label"].value_counts().reindex([0,1,2],fill_value=0)
if (counts < CONFIG["classification"]["minimum_shops_per_class"]).any(): raise ValueError(f"Need at least five shops per class; found {counts.to_dict()}")
test_n = max(3,int(np.ceil(len(data)*CONFIG["classification"]["test_size"])))
train,test = train_test_split(data,test_size=test_n,random_state=CONFIG["project"]["random_state"],stratify=data["Target_Label"])
model = RandomForestClassifier(n_estimators=CONFIG["classification"]["n_estimators"],random_state=CONFIG["project"]["random_state"],class_weight="balanced").fit(train[features],train["Target_Label"])
dummy = DummyClassifier(strategy="most_frequent").fit(train[features],train["Target_Label"])
prediction = model.predict(test[features]); probability = model.predict_proba(test[features]).max(axis=1)
macro = precision_recall_fscore_support(test["Target_Label"],prediction,average="macro",zero_division=0)
metrics = {"input_fingerprint":INPUT_FINGERPRINT,"accuracy":accuracy_score(test["Target_Label"],prediction),"baseline_accuracy":accuracy_score(test["Target_Label"],dummy.predict(test[features])),"macro_precision":macro[0],"macro_recall":macro[1],"macro_f1":macro[2],"weighted_f1":precision_recall_fscore_support(test["Target_Label"],prediction,average="weighted",zero_division=0)[2],"target_accuracy":CONFIG["classification"]["accuracy_target"]}
metrics["pass_accuracy_requirement"] = metrics["accuracy"] > metrics["target_accuracy"]
cm = confusion_matrix(test["Target_Label"],prediction,labels=[0,1,2])
fig,ax=plt.subplots(); sns.heatmap(cm,annot=True,fmt="d",xticklabels=[0,1,2],yticklabels=[0,1,2],ax=ax); ax.set(xlabel="Predicted",ylabel="Actual",title="Confusion matrix"); fig.tight_layout(); fig.savefig(OUTPUT/"figures"/"confusion_matrix.png",dpi=160); plt.show(); plt.close(fig)
importance = pd.DataFrame({"Feature":features,"Importance":model.feature_importances_}).sort_values("Importance",ascending=False)
fig,ax=plt.subplots(); sns.barplot(data=importance,x="Importance",y="Feature",ax=ax); fig.tight_layout(); fig.savefig(OUTPUT/"figures"/"classification_feature_importance.png",dpi=160); plt.show(); plt.close(fig)
result = pd.DataFrame({"Shop_ID":test["Shop_ID"],"Actual_Label":test["Target_Label"],"Predicted_Label":prediction,"Predicted_Probability":probability})
result.to_csv(OUTPUT/"reports"/"Classification_Result.csv",index=False); importance.to_csv(OUTPUT/"reports"/"Classification_Feature_Importance.csv",index=False)
(OUTPUT/"reports"/"classification_metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")
joblib.dump(model,OUTPUT/"models"/"random_forest_shop_reputation.joblib")
print(classification_report(test["Target_Label"],prediction,labels=[0,1,2],zero_division=0)); display(pd.DataFrame([metrics])); display(importance)